# Tractor-Mix GWAS

**Pipeline reference:** https://github.com/Atkinson-Lab/Tractor-Mix 

In [ ]:
import os
import subprocess
import time
import tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

In [ ]:
# Rscript path
RSCRIPT = "/path/to/home/miniconda3/envs/tractormix/bin/Rscript"

# Base directories
BASE_DIR       = "/path/to/home/DATA"
ANALYSIS_DIR   = os.path.join(BASE_DIR, "ANALYSIS")
IMPUTED_DIR    = os.path.join(BASE_DIR, "IMPUTED")
GRM_DIR        = os.path.join(ANALYSIS_DIR, "GRM")
PCA_DIR        = os.path.join(ANALYSIS_DIR, "PCA")
TRACTORMIX_DIR = "/path/to/home/TRACTOR-MIX/Tractor-Mix"
OUTPUT_DIR     = os.path.join(ANALYSIS_DIR, "TRACTORMIX_RESULTS")

# Covariate file (columns: ID, SEX, AGE, GCTA_PC1-10, DISEASE)
COVAR_PATH = os.path.join(PCA_DIR, "CATPD.step1.tsv")

# Plink files (for GRM computation)
PLINK_PREFIX        = os.path.join(GRM_DIR, "ALLCHR_OnlyTyped")
PLINK_PRUNED_PREFIX = os.path.join(GRM_DIR, "ALLCHR_OnlyTyped_pruned")
PRUNED_SNPS_PREFIX  = os.path.join(GRM_DIR, "pruned_snps")
GRM_PREFIX          = os.path.join(GRM_DIR, "ALLCHR_OnlyTyped")

# Dosage file pattern (per chromosome)
DOSAGE_PATTERN   = os.path.join(IMPUTED_DIR, "ALLCHR_OnlyTyped.{chr}.anc{anc}.dosage.txt")
HAPCOUNT_PATTERN = os.path.join(IMPUTED_DIR, "ALLCHR_OnlyTyped.{chr}.anc{anc}.hapcount.txt")

# Tractor-Mix R scripts
SCORE_SCRIPT      = os.path.join(TRACTORMIX_DIR, "TractorMix.score.R")
SCORE_COND_SCRIPT = os.path.join(TRACTORMIX_DIR, "TractorMix.score_cond.R")

# Null model will be saved/loaded as RDS
NULL_MODEL_RDS = os.path.join(OUTPUT_DIR, "Model_Null.rds")

# Chromosomes to process
CHROMOSOMES = list(range(1, 23))

# Number of ancestries
N_ANC = 6

# Resource limits
MAX_THREADS      = 8   # for plink2/gcta and within each R process
MAX_PARALLEL     = 4   # chromosomes to run in parallel
THREADS_PER_CHR  = 2   # R threads per chromosome (MAX_PARALLEL * THREADS_PER_CHR <= total CPUs)
AC_THRESHOLD     = 20

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("All paths configured.")
print(f"Rscript:     {RSCRIPT}")
print(f"Chromosomes: {CHROMOSOMES}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Parallelism: {MAX_PARALLEL} chromosomes x {THREADS_PER_CHR} threads each")


In [ ]:
# Verify Rscript and define scripts
def run_cmd(cmd, desc="", cwd=None):
    """Run a shell command with timing."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {desc}")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    elapsed = time.time() - start
    if result.returncode != 0:
        print(f"  STDERR: {result.stderr[:500]}")
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    print(f"  Done in {elapsed:.1f}s")
    return result


def run_r(script_str, desc="", timeout=7200):
    """Run an R script string via subprocess Rscript."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {desc}")
    start = time.time()
    
    with tempfile.NamedTemporaryFile(mode="w", suffix=".R", delete=False, dir=OUTPUT_DIR) as f:
        f.write(script_str)
        script_path = f.name
    
    try:
        result = subprocess.run(
            [RSCRIPT, script_path],
            capture_output=True, text=True, timeout=timeout
        )
        elapsed = time.time() - start
        
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"R STDERR:\n{result.stderr[:1000]}")
            raise RuntimeError(f"R script failed after {elapsed:.1f}s")
        
        print(f"  Done in {elapsed/60:.1f} min")
        return result
    finally:
        os.unlink(script_path)


def check_files_exist(file_list, label=""):
    missing = [f for f in file_list if not os.path.exists(f)]
    if missing:
        print(f"WARNING — {label} missing:")
        for f in missing:
            print(f"  {f}")
        return False
    print(f"{label}: all {len(file_list)} files found.")
    return True


def get_dosage_files(chrom):
    return [DOSAGE_PATTERN.format(chr=chrom, anc=a) for a in range(N_ANC)]


def get_hapcount_files(chrom):
    return [HAPCOUNT_PATTERN.format(chr=chrom, anc=a) for a in range(N_ANC)]


# Verify Rscript works with GMMAT
result = subprocess.run(
    [RSCRIPT, "-e", "library(GMMAT); library(Matrix); library(data.table); cat('All R packages OK\n')"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"ERROR: {result.stderr}")
    raise RuntimeError("Rscript cannot load required packages")

print("Helpers defined, Rscript verified.")


In [ ]:
# Verify all input files
check_files_exist([COVAR_PATH], "Covariate file")
check_files_exist([SCORE_SCRIPT, SCORE_COND_SCRIPT], "TractorMix scripts")

plink_files = [f"{PLINK_PREFIX}.{ext}" for ext in ["bed", "bim", "fam"]]
check_files_exist(plink_files, "Plink files")

for chrom in CHROMOSOMES:
    check_files_exist(get_dosage_files(chrom), f"Chr{chrom} dosage")


In [ ]:
# Compute GRM (QC + prune + GCTA)
start = time.time()

# Step 1: QC filter + LD prune
run_cmd([
    "plink2", "--bfile", PLINK_PREFIX,
    "--maf", "0.01", "--geno", "0.02", "--hwe", "1e-6",
    "--indep-pairwise", "500", "50", "0.2",
    "--out", PRUNED_SNPS_PREFIX,
    "--threads", str(MAX_THREADS)
], desc="Step 1: QC + LD prune")

# Step 2: Extract pruned SNPs
run_cmd([
    "plink2", "--bfile", PLINK_PREFIX,
    "--extract", f"{PRUNED_SNPS_PREFIX}.prune.in",
    "--make-bed",
    "--out", PLINK_PRUNED_PREFIX,
    "--threads", str(MAX_THREADS)
], desc="Step 2: Extract pruned SNPs")

# Step 3: GCTA GRM in parts
N_PARTS = 10
for i in range(1, N_PARTS + 1):
    run_cmd([
        "gcta64", "--bfile", PLINK_PRUNED_PREFIX,
        "--make-grm-part", str(N_PARTS), str(i),
        "--out", GRM_PREFIX,
        "--threads", str(MAX_THREADS)
    ], desc=f"Step 3: GCTA GRM part {i}/{N_PARTS}")

# Merge parts
for ext in ["grm.id", "grm.bin", "grm.N.bin"]:
    part_files = sorted([
        f for f in os.listdir(GRM_DIR)
        if f.startswith(os.path.basename(GRM_PREFIX) + f".part_{N_PARTS}_") and f.endswith(ext)
    ])
    with open(f"{GRM_PREFIX}.{ext}", "wb") as outf:
        for pf in part_files:
            with open(os.path.join(GRM_DIR, pf), "rb") as inf:
                outf.write(inf.read())
    print(f"  Merged {len(part_files)} parts → {os.path.basename(GRM_PREFIX)}.{ext}")

print(f"\nGRM done in {(time.time()-start)/60:.1f} minutes")

In [ ]:
# Load GRM, fit null model, save as RDS

run_r(f'''
library(GMMAT)
library(Matrix)
library(data.table)

Sys.setenv(OMP_NUM_THREADS = {MAX_THREADS})
Sys.setenv(OPENBLAS_NUM_THREADS = {MAX_THREADS})
Sys.setenv(MKL_NUM_THREADS = {MAX_THREADS})

# --- Load GRM ---
cat("Loading GRM...\n")
grm_ids <- fread("{GRM_PREFIX}.grm.id", header = FALSE)
ids <- as.character(grm_ids$V2)
n <- length(ids)

grm_bin <- readBin("{GRM_PREFIX}.grm.bin", what = numeric(),
                   n = n * (n + 1) / 2, size = 4)

GRM <- matrix(0, n, n)
k <- 1
for (i in 1:n) {{
    GRM[i, 1:i] <- grm_bin[k:(k + i - 1)]
    GRM[1:i, i] <- grm_bin[k:(k + i - 1)]
    k <- k + i
}}
rownames(GRM) <- ids
colnames(GRM) <- ids
cat("GRM loaded. Dim:", dim(GRM), "\n")

# --- Load covariates ---
cat("Loading covariates...\n")
df <- as.data.frame(fread("{COVAR_PATH}", header = TRUE))
df$ID <- as.character(df$ID)

# --- Align ---
common_ids <- intersect(df$ID, ids)
cat("Samples in covar:", nrow(df), "\n")
cat("Samples in GRM:  ", length(ids), "\n")
cat("Overlap:         ", length(common_ids), "\n")

df  <- df[match(common_ids, df$ID), ]
GRM <- GRM[common_ids, common_ids]

stopifnot(all(df$ID == rownames(GRM)))
cat("DISEASE prevalence:", mean(df$DISEASE, na.rm=TRUE), "\n")

# --- Fit null model ---
cat("Fitting null model...\n")
Model_Null <- glmmkin(
    fixed  = DISEASE ~ SEX + AGE_AAO + GCTA_PC1 + GCTA_PC2 + GCTA_PC3 + GCTA_PC4 +
             GCTA_PC5 + GCTA_PC6 + GCTA_PC7 + GCTA_PC8 + GCTA_PC9 + GCTA_PC10,
    data   = df,
    id     = "ID",
    kins   = GRM,
    family = binomial(),
    verbose = TRUE
)

cat("\nNull model fitted.\n")
cat("Fixed effect estimates:\n")
print(Model_Null$coefficients)

# --- Save ---
saveRDS(Model_Null, "{NULL_MODEL_RDS}")
cat("\nModel saved to: {NULL_MODEL_RDS}\n")
''', desc="Fitting null model (this may take 15-30 min)")


In [ ]:
# Run TractorMix per chromosome in parallel
def run_tractormix_chrom(chrom):
    """Run TractorMix for one chromosome via Rscript subprocess."""
    
    start = time.time()
    dosage = get_dosage_files(chrom)
    outfile = os.path.join(OUTPUT_DIR, f"tractormix_uncond_chr{chrom}.tsv")
    
    # Check dosage files
    for f in dosage:
        if not os.path.exists(f):
            return f"Chr{chrom}: FAILED — missing {os.path.basename(f)}"
    
    infiles_r = 'c(' + ', '.join([f'"{f}"' for f in dosage]) + ')'
    
    r_script = f'''
library(GMMAT)
library(Matrix)
library(data.table)
library(doParallel)
library(foreach)
library(dplyr)
library(abind)

Sys.setenv(OMP_NUM_THREADS = {THREADS_PER_CHR})
Sys.setenv(OPENBLAS_NUM_THREADS = {THREADS_PER_CHR})
Sys.setenv(MKL_NUM_THREADS = {THREADS_PER_CHR})
registerDoParallel(cores = {THREADS_PER_CHR})

cat("Chr{chrom}: Loading null model...\n")
Model_Null <- readRDS("{NULL_MODEL_RDS}")

cat("Chr{chrom}: Running TractorMix...\n")
source("{SCORE_SCRIPT}")
TractorMix.score(
    obj          = Model_Null,
    infiles      = {infiles_r},
    outfiles     = "{outfile}",
    AC_threshold = {AC_THRESHOLD}
)

cat("Chr{chrom}: Done.\n")
'''
    
    script_path = os.path.join(OUTPUT_DIR, f"run_chr{chrom}.R")
    with open(script_path, "w") as f:
        f.write(r_script)
    
    try:
        result = subprocess.run(
            [RSCRIPT, script_path],
            capture_output=True, text=True, timeout=7200
        )
        elapsed = (time.time() - start) / 60
        
        if result.returncode != 0:
            return f"Chr{chrom}: FAILED after {elapsed:.1f} min — {result.stderr[:300]}"
        
        if os.path.exists(outfile):
            with open(outfile) as f:
                n_lines = sum(1 for _ in f) - 1
            return f"Chr{chrom}: {n_lines} variants in {elapsed:.1f} min"
        else:
            return f"Chr{chrom}: FAILED — no output file produced"
    
    except subprocess.TimeoutExpired:
        return f"Chr{chrom}: TIMEOUT after 120 min"
    except Exception as e:
        return f"Chr{chrom}: ERROR — {str(e)[:200]}"
    finally:
        if os.path.exists(script_path):
            os.remove(script_path)


# Run all chromosomes in parallel
start_all = time.time()
results = []

print(f"Running {len(CHROMOSOMES)} chromosomes, {MAX_PARALLEL} at a time...")
print(f"Each process: {THREADS_PER_CHR} threads")
print()

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as executor:
    futures = {executor.submit(run_tractormix_chrom, c): c for c in CHROMOSOMES}
    
    for future in as_completed(futures):
        chrom = futures[future]
        result = future.result()
        results.append((chrom, result))
        print(result)

total_min = (time.time() - start_all) / 60
print(f"\nAll chromosomes done in {total_min:.1f} minutes")

# Sort and print summary
results.sort(key=lambda x: x[0])
print("\nSummary:")
for chrom, r_str in results:
    print(f"  {r_str}")


In [ ]:
# Concatenate all chromosome results
all_results = []
for chrom in CHROMOSOMES:
    outfile = os.path.join(OUTPUT_DIR, f"tractormix_uncond_chr{chrom}.tsv")
    if os.path.exists(outfile):
        df_chr = pd.read_csv(outfile, sep="\t")
        if "CHR" not in df_chr.columns:
            df_chr.insert(0, "CHR", chrom)
        all_results.append(df_chr)
        print(f"Chr{chrom}: {len(df_chr)} variants")
    else:
        print(f"Chr{chrom}: MISSING")

if all_results:
    df_all = pd.concat(all_results, ignore_index=True)
    
    combined_outfile = os.path.join(OUTPUT_DIR, "tractormix_uncond_allchr.tsv")
    df_all.to_csv(combined_outfile, sep="\t", index=False)
    
    print(f"\nTotal variants: {len(df_all)}")
    print(f"Combined results: {combined_outfile}")
    
    # Top hits
    if "P" in df_all.columns:
        print("\nTop 20 hits by joint P-value:")
        top = df_all.nsmallest(20, "P")
        display_cols = [c for c in ["CHR", "POS", "ID", "P", "Eff_anc0", "Eff_anc1", 
                                     "Pval_anc0", "Pval_anc1"] if c in top.columns]
        print(top[display_cols].to_string(index=False))
else:
    print("No results to concatenate!")


In [ ]:
# Check output files
print("Output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f:50s} {size_mb:8.1f} MB")

print(f"\nNull model RDS: {os.path.exists(NULL_MODEL_RDS)}")
print(f"Combined TSV:   {os.path.exists(os.path.join(OUTPUT_DIR, 'tractormix_uncond_allchr.tsv'))}")
